Dependency Parsing

# Phần 1: Cài đặt

In [3]:
# Cài đặt Spacy
!pip install -U spacy
# Tải mô hình ngôn ngữ tiếng Anh
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Phần 2: Phân tích và trực quan hóa

## 2.1 Tải mô hình và phân tích câu

In [4]:
import spacy
from spacy import displacy

In [7]:
# Sử dụng en_core_web_sm vì nó đã được cài đặt và cung cấp các tính năng cơ bản cho phân tích cú pháp
nlp = spacy.load("en_core_web_sm")
# Câu ví dụ
text = "The quick brown fox jumps over the lazy dog."
# Phân tích câu với pipeline của spaCy
doc = nlp(text)

## 2.2. Trực quan hóa cây phụ thuộc

In [8]:
# Khởi chạy server tại http://127.0.0.1:5000
displacy.serve(doc, style="dep")

/usr/local/lib/python3.12/dist-packages/spacy/displacy/__init__.py:108: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


# Phần 3: Truy cập các thành phần trong cây phụ thuộc

In [9]:
# Lấy một câu khác để phân tích
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
# In ra thông tin của từng token
print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)

for token in doc:
    # Trích xuất các thuộc tính
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------
Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | nsubj      | startup      | VERB     | []
startup      | ccomp      | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | VERB     | ['billion']
$            | quantmod   | billion      | NUM      | []
1            | compound   | billion      | NUM      | []
billion      | pobj       | for          | ADP      | ['$', '1']


# Phần 4: Duyệt cây phụ thuộc để trích xuất thông tin

## 4.1. Bài toán: Tìm chủ ngữ và tân ngữ của một động từ

In [11]:
text = "The cat chased the mouse and the dog watched them."
doc = nlp(text)
for token in doc:
# Chỉ tìm các động từ
    if token.pos_ == "VERB":
        verb = token.text
        subject = ""
        obj = ""
    # Tìm chủ ngữ (nsubj) và tân ngữ (dobj) trong các con của động từ
        for child in token.children:
            if child.dep_ == "nsubj":
                subject = child.text
            if child.dep_ == "dobj":
                obj = child.text
        if subject and obj:
            print(f"Found Triplet: ({subject}, {verb}, {obj})")

Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


## 4.2. Bài toán: Tìm các tính từ bổ nghĩa cho một danh từ

In [13]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc = nlp(text)
for token in doc:
    # Chỉ tìm các danh từ
    if token.pos_ == "NOUN":
        adjectives = []
        # Tìm các tính từ bổ nghĩa (amod) trong các con của danh từ
        for child in token.children:
            if child.dep_ == "amod":
                adjectives.append(child.text)
        if adjectives:
            print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")

Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


# Phần 5: Bài tập tự luyện

## Bài 1: Tìm động từ chính của câu

In [17]:
def find_main_verb(doc):
    """
    Tìm động từ chính của câu (ROOT verb)

    Args:
        doc: Đối tượng Doc của spaCy

    Returns:
        Token là động từ chính (có dep_ == "ROOT")
    """
    for token in doc:
        if token.dep_ == "ROOT":
            return token
    return None

# Test hàm với các câu ví dụ
test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Apple is looking at buying U.K. startup for $1 billion",
    "The cat chased the mouse and the dog watched them.",
    "I love learning Natural Language Processing."
]

for sentence in test_sentences:
    doc = nlp(sentence)
    main_verb = find_main_verb(doc)

    if main_verb:
        print(f"\nCâu: {sentence}")
        print(f" Động từ chính: '{main_verb.text}'")
        print(f" POS tag: {main_verb.pos_}")
        print(f" Lemma: {main_verb.lemma_}")
        print(f" Dependency: {main_verb.dep_}")
    else:
        print(f"\nCâu: {sentence}")
        print(" Không tìm thấy động từ chính!")


Câu: The quick brown fox jumps over the lazy dog.
 Động từ chính: 'jumps'
 POS tag: VERB
 Lemma: jump
 Dependency: ROOT

Câu: Apple is looking at buying U.K. startup for $1 billion
 Động từ chính: 'looking'
 POS tag: VERB
 Lemma: look
 Dependency: ROOT

Câu: The cat chased the mouse and the dog watched them.
 Động từ chính: 'chased'
 POS tag: VERB
 Lemma: chase
 Dependency: ROOT

Câu: I love learning Natural Language Processing.
 Động từ chính: 'love'
 POS tag: VERB
 Lemma: love
 Dependency: ROOT


## Bài 2: Trích xuất các cụm danh từ (Noun Chunks)

### Hàm trích xuất

In [29]:
def extract_noun_chunks(doc):
    """
    Trích xuất các cụm danh từ từ câu

    Một cụm danh từ bao gồm:
    - Danh từ chính (head noun)
    - Các từ bổ nghĩa: det (determiner), amod (adjective), compound (noun compound), etc.

    Args:
        doc: Đối tượng Doc của spaCy

    Returns:
        List các cụm danh từ (mỗi cụm là một string)
    """
    noun_chunks = []
    processed_tokens = set()

    for token in doc:
        # Tìm các danh từ hoặc đại từ làm head
        if token.pos_ in ["NOUN", "PROPN", "PRON"] and token.i not in processed_tokens:
            chunk_tokens = []

            # Thu thập các từ bổ nghĩa (children) của danh từ
            for child in token.children:
                if child.dep_ in ["det", "amod", "compound", "nummod", "poss"]:
                    chunk_tokens.append(child)
                    processed_tokens.add(child.i)

            # Thêm danh từ chính
            chunk_tokens.append(token)
            processed_tokens.add(token.i)

            # Sắp xếp theo vị trí trong câu
            chunk_tokens.sort(key=lambda x: x.i)

            # Tạo chuỗi từ các token
            chunk_text = " ".join([t.text for t in chunk_tokens])
            noun_chunks.append({
                'text': chunk_text,
                'root': token.text
            })

    return noun_chunks


In [30]:

# Test hàm với các câu ví dụ
test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Apple is looking at buying U.K. startup for $1 billion",
    "The big fluffy white cat sleeps on the warm comfortable mat."
]


for sentence in test_sentences:
    doc = nlp(sentence)

    # So sánh với noun_chunks có sẵn của spaCy
    spacy_chunks = [chunk.text for chunk in doc.noun_chunks]
    my_chunks = extract_noun_chunks(doc)

    print(f"\nCâu: {sentence}")
    print('Spacy chunks')
    for chunk in spacy_chunks:
        print(f"   {chunk}")

    print(f"My chunks:")
    for chunk in my_chunks:
        print(f"   {chunk['text']} (root: {chunk['root']})")


Câu: The quick brown fox jumps over the lazy dog.
Spacy chunks
   The quick brown fox
   the lazy dog
My chunks:
   The quick brown fox (root: fox)
   the lazy dog (root: dog)

Câu: Apple is looking at buying U.K. startup for $1 billion
Spacy chunks
   Apple
   U.K.
My chunks:
   Apple (root: Apple)
   U.K. (root: U.K.)

Câu: The big fluffy white cat sleeps on the warm comfortable mat.
Spacy chunks
   The big fluffy white cat
   the warm comfortable mat
My chunks:
   The big fluffy white cat (root: cat)
   the warm comfortable mat (root: mat)


## Bài 3: Tìm đường đi ngắn nhất trong cây

In [44]:
def get_path_to_root(token):
    """
    Tìm đường đi từ một token bất kỳ lên đến gốc (ROOT) của cây

    Args:
        token: Token cần tìm đường đi

    Returns:
        List các token trên đường đi từ token hiện tại đến ROOT
    """
    path = [token]
    current = token

    # Duyệt lên theo head cho đến khi gặp ROOT
    while current.dep_ != "ROOT":
        current = current.head
        path.append(current)
    path.append('ROOT')
    return path

def get_distance_to_root(token):
    """
    Tính khoảng cách (số bước) từ token đến ROOT

    Args:
        token: Token cần tính khoảng cách

    Returns:
        Số bước từ token đến ROOT
    """
    path = get_path_to_root(token)
    return len(path) - 2  # Không tính token hiện tại + ROOT

In [45]:
# Test hàm với câu ví dụ
test_sentence = "The quick brown fox jumps over the lazy dog."
doc = nlp(test_sentence)

print(f"\nCâu: {test_sentence}\n")

# Chọn một số token để demo
demo_tokens = [doc[0], doc[2], doc[8]]  # "The", "brown", "dog"

for token in demo_tokens:
    path = get_path_to_root(token)
    distance = get_distance_to_root(token)
    print(f'Đường đi của {token} đến ROOT là')
    pp = ''
    for p in path:
      pp += str(p) + ' -> '
    print(pp[:-4])
    print(f'Khoảng cách từ {token} đến ROOT là {distance}')


Câu: The quick brown fox jumps over the lazy dog.

Đường đi của The đến ROOT là
The -> fox -> jumps -> ROOT
Khoảng cách từ The đến ROOT là 2
Đường đi của brown đến ROOT là
brown -> fox -> jumps -> ROOT
Khoảng cách từ brown đến ROOT là 2
Đường đi của dog đến ROOT là
dog -> over -> jumps -> ROOT
Khoảng cách từ dog đến ROOT là 2
